# Model D (full): ITA-guided L*-CLAHE + two-stage decoupled training + Focal Loss

This is the complete Model D described in the paper.

| Stage | What | Status |
|---|---|---|
| Stage 1: localization | YOLO26n trained on ITA-adaptive L*-CLAHE images (normal loss) | **Done**: D2 weights, seeds 42, 43, 44 |
| Stage 2: fine-grained classification | Start from Stage 1, **freeze backbone layers 0–10**, train with **Focal Loss** (α = 0.25, γ = 2.0) | **This notebook** |

**Run All once** (or headless, see the last cell). About 30–45 min per seed, 3 seeds.
If it stops, Run All again: finished seeds are skipped and an interrupted one resumes.

Rules kept for a fair comparison:
- Same data, split, image size (640), batch (8) and seeds as Models A, B, C.
- The best epoch is chosen on the **validation** set; the **test** set is only used at the end, once.
- All three seeds are reported (mean ± SD); no seed is picked by its test score.


## Setup

In [1]:
import os, sys, shutil
from pathlib import Path
os.environ.setdefault('MPLBACKEND', 'Agg')

HERE = Path.cwd().resolve()
if not (HERE / 'sop_analysis.py').is_file():
    HERE = HERE / 'final'
assert (HERE / 'sop_analysis.py').is_file(), 'Open this notebook from ablation_training/final.'
ROOT = HERE.parent
DATA_YAML = ROOT / 'datasets' / 'ablation_yolov26' / 'ModelD2_lesion_ita' / 'data.yaml'
WEIGHTS_ROOT = ROOT / 'weights' / 'ablation_split'
RUNS = ROOT / 'runs' / 'ablation_split' / 'training'

SEEDS = (42, 43, 44)
FOCAL_ALPHA = 0.25
FOCAL_GAMMA = 2.0
FREEZE = 11
EPOCHS = 150
PATIENCE = 50
LR0 = 0.0002
BATCH, IMG_SIZE = 8, 640

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
assert torch.cuda.is_available(), 'No GPU: select the TOOL-26 (.venv) kernel.'
assert DATA_YAML.is_file(), f'Missing D2 dataset: {DATA_YAML}'
for seed in SEEDS:
    stage1 = WEIGHTS_ROOT / f'best_ModelD2_lesion_ita_seed{seed}.pt'
    assert stage1.is_file(), f'Missing Stage 1 weights: {stage1}'
print('Stage 1 weights found for seeds', SEEDS)

GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Stage 1 weights found for seeds (42, 43, 44)


## Focal Loss (α = 0.25, γ = 2.0)
Replaces the class loss and checks that it is really used.

In [2]:
import torch.nn as nn
import torch.nn.functional as F
from ultralytics.utils import loss as uloss


class ElementwiseFocalLoss(nn.Module):
    def __init__(self, gamma: float, alpha: float):
        super().__init__()
        self.gamma, self.alpha = gamma, alpha

    def forward(self, pred, label):
        loss = F.binary_cross_entropy_with_logits(pred, label, reduction='none')
        prob = pred.sigmoid()
        p_t = label * prob + (1 - label) * (1 - prob)
        loss = loss * (1.0 - p_t) ** self.gamma
        if self.alpha > 0:
            loss = loss * (label * self.alpha + (1 - label) * (1 - self.alpha))
        return loss


if not hasattr(uloss.v8DetectionLoss, '_init_without_focal'):
    uloss.v8DetectionLoss._init_without_focal = uloss.v8DetectionLoss.__init__

def _init_with_focal(self, *args, **kwargs):
    uloss.v8DetectionLoss._init_without_focal(self, *args, **kwargs)
    self.bce = ElementwiseFocalLoss(FOCAL_GAMMA, FOCAL_ALPHA)

uloss.v8DetectionLoss.__init__ = _init_with_focal

pred, label = torch.randn(4, 50, 8), torch.rand(4, 50, 8)
ours = ElementwiseFocalLoss(FOCAL_GAMMA, FOCAL_ALPHA)(pred, label).mean(1).sum()
ref = uloss.FocalLoss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA)(pred, label)
assert torch.allclose(ours, ref, atol=1e-6), (ours, ref)

from ultralytics import YOLO
from ultralytics.cfg import get_cfg
m = YOLO(str(WEIGHTS_ROOT / 'best_ModelD2_lesion_ita_seed42.pt')).model
m.args = get_cfg()
crit = m.init_criterion()
heads = [crit.one2many, crit.one2one] if hasattr(crit, 'one2many') else [crit]
assert all(isinstance(h.bce, ElementwiseFocalLoss) for h in heads)
del m, crit
print(f'Focal Loss active (alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA}) in {len(heads)} detection head(s).')

Focal Loss active (alpha=0.25, gamma=2.0) in 2 detection head(s).


## Stage 2 training (frozen backbone + Focal Loss), seeds 42, 43, 44

In [3]:
def checkpoint_state(checkpoint: Path) -> str:
    try:
        epoch = torch.load(checkpoint, map_location='cpu', weights_only=False).get('epoch', 0)
    except Exception:
        return 'corrupt'
    return 'finished' if epoch == -1 else 'resumable'


def train_stage2(seed: int) -> Path:
    name = f'ModelD_full_seed{seed}'
    destination = WEIGHTS_ROOT / f'best_ModelD_full_seed{seed}.pt'
    if destination.is_file():
        print(f'Seed {seed}: already finished -> {destination.name}')
        return destination
    run = RUNS / name
    last = run / 'weights' / 'last.pt'
    state = checkpoint_state(last) if last.is_file() else 'missing'
    if state == 'corrupt':
        print(f'{last} is unreadable; restarting seed {seed}.')
        shutil.rmtree(run)
        state = 'missing'
    if state == 'resumable':
        print(f'Seed {seed}: resuming from {last}')
        YOLO(str(last)).train(resume=True)
    elif state == 'missing':
        print(f'Seed {seed}: Stage 2 from best_ModelD2_lesion_ita_seed{seed}.pt '
              f'(freeze layers 0-{FREEZE - 1}, Focal Loss)')
        YOLO(str(WEIGHTS_ROOT / f'best_ModelD2_lesion_ita_seed{seed}.pt')).train(
            data=str(DATA_YAML), epochs=EPOCHS, patience=PATIENCE, imgsz=IMG_SIZE, batch=BATCH,
            seed=seed, deterministic=True, freeze=FREEZE, optimizer='AdamW', lr0=LR0, warmup_epochs=1,
            project=str(RUNS), name=name, exist_ok=True, plots=True, verbose=True,
        )
    best = run / 'weights' / 'best.pt'
    assert best.is_file(), f'Missing validation-selected weights: {best}'
    shutil.copy2(best, destination)
    return destination


for seed in SEEDS:
    train_stage2(seed)
print('Stage 2 finished for seeds', SEEDS)


Seed 42: Stage 2 from best_ModelD2_lesion_ita_seed42.pt (freeze layers 0-10, Focal Loss)


New https://pypi.org/project/ultralytics/8.4.168 available 😃 Update with 'pip install -U ultralytics'


Ultralytics 8.4.163 🚀 Python-3.14.4 torch-2.14.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 3770MiB)


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/eney/Documents/TOOL2026/ablation_training/datasets/ablation_yolov26/ModelD2_lesion_ita/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=11, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/eney/Documents/TOOL2026/ablation_training/weights/ablation_split/best_ModelD2_lesion_ita_seed42.pt, momentum=0.937,


                   from  n    params  module                                       arguments                     


...

           Tinea pedis         11         11      0.555      0.727       0.71      0.441


              Impetigo         58        260      0.536      0.485       0.48      0.189


Speed: 0.2ms preprocess, 2.2ms inference, 0.0ms loss, 4.9ms postprocess per image


Results saved to /home/eney/Documents/TOOL2026/ablation_training/runs/ablation_split/training/ModelD_full_seed44


Stage 2 finished for seeds (42, 43, 44)


## Validation: Stage 1 (D2) vs full D
This is the decision table (the test set is not used here).

In [4]:
import pandas as pd
rows = []
for seed in SEEDS:
    for label, weights in (('D2 (Stage 1 only)', f'best_ModelD2_lesion_ita_seed{seed}.pt'),
                           ('Full D (Stage 2 + Focal)', f'best_ModelD_full_seed{seed}.pt')):
        m = YOLO(str(WEIGHTS_ROOT / weights)).val(data=str(DATA_YAML), split='val', batch=BATCH, imgsz=IMG_SIZE,
                                                    workers=0, plots=False, verbose=False)
        rows.append({'model': label, 'seed': seed, 'mAP50': 100 * m.box.map50, 'mAP50_95': 100 * m.box.map,
                     'precision': 100 * m.box.mp, 'recall': 100 * m.box.mr})
val = pd.DataFrame(rows)
val.to_csv(HERE / 'full_D_validation.csv', index=False)
summary = val.groupby('model')[['mAP50', 'mAP50_95', 'precision', 'recall']].agg(['mean', 'std']).round(1)
print(summary.to_string())

Ultralytics 8.4.163 🚀 Python-3.14.4 torch-2.14.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 3770MiB)


YOLO26n summary (fused): 120 layers, 2,376,396 parameters, 0 gradients, 5.3 GFLOPs


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6976.9±1151.9 MB/s, size: 605.3 KB)


val: Scanning /home/eney/Documents/TOOL2026/ablation_training/datasets/ablation_yolov26/ModelD2_lesion_ita/labels/val.cache... 397 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 397/397 185.0Mit/s 0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 4% ──────────── 2/50 3.9it/s 0.2s<12.3s

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 96% ━━━━━━━━━━━╸ 48/50 9.6it/s 5.2s<0.2s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 50/50 9.4it/s 5.3s

                   all        397       3116      0.532      0.449      0.439      0.212


Speed: 0.2ms preprocess, 3.6ms inference, 0.0ms loss, 3.4ms postprocess per image


                         mAP50      mAP50_95      precision      recall     
                          mean  std     mean  std      mean  std   mean  std
model                                                                       
D2 (Stage 1 only)         43.2  0.3     20.2  0.4      51.9  3.5   45.0  1.8
Full D (Stage 2 + Focal)  44.2  0.9     21.1  0.1      53.5  1.6   44.9  1.0


## Test set: SOP tables for A, B, C and the full D

In [5]:
import importlib
os.environ['SOP_MODEL_D'] = 'full'
sys.path.insert(0, str(HERE))
import sop_analysis
importlib.reload(sop_analysis)
sop_analysis.main()
print((sop_analysis.OUT / 'SOP_RESULTS.md').read_text(encoding='utf-8'))

Model A (Baseline YOLOv26) seed 42: evaluating the 200 test images...


Ultralytics 8.4.163 🚀 Python-3.14.4 torch-2.14.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 3770MiB)


YOLO26n summary (fused): 120 layers, 2,376,396 parameters, 0 gradients, 5.3 GFLOPs


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 400.2±164.8 MB/s, size: 251.4 KB)


val: Scanning /home/eney/Documents/TOOL2026/ablation_training/datasets/ablation_yolov26/ModelA_raw/labels/test.cache... 200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 200/200 93.2Mit/s 0.0s

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 92% ━━━━━━━━━━━─ 23/25 8.6it/s 2.4s<0.2s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 9.6it/s 2.6s

                   all        200       1435      0.453      0.441      0.399      0.203


Speed: 0.2ms preprocess, 3.6ms inference, 0.0ms loss, 3.5ms postprocess per image



Done. Results in /home/eney/Documents/TOOL2026/ablation_training/final/sop_results_full_D
# Statement of the Problem: results

Test set: 200 images (never used in training or calibration). Models A, B, C, D = A Baseline YOLOv26, B RGB CLAHE, C Fixed L*-CLAHE (C'), D ITA-guided L*-CLAHE + two-stage + Focal Loss (full D).
mAP, precision, recall and F1 tables are the official Ultralytics test evaluation (P and R at the confidence that maximizes F1). Per-image scores for the tests and the confusion use confidence >= 0.25 and IoU >= 0.5. Paired tests use seed 42 for every model; C and D also show the mean ± SD of seeds 42-44.

## SOP 1: localization (mAP@50, mAP@50-95) and SOP 2: classification (P, R, F1)

| Model | mAP@50 | mAP@50-95 | Precision | Recall | F1 |
|---|---|---|---|---|---|
| A Baseline YOLOv26 (seed 42) | 42.6 | 19.0 | 45.2 | 47.9 | 46.5 |
| B RGB CLAHE (seed 42) | 43.4 | 18.8 | 55.4 | 42.8 | 48.3 |
| C Fixed L*-CLAHE (C') (seed 42) | 42.2 | 20.7 | 47.8 | 43.6 | 45.6 |
| D I

## Run headless (recommended: less RAM, no VS Code)

In a terminal, from `ablation_training/final`:

```bash
setsid nohup ../../TOOL-26/.venv/bin/jupyter nbconvert --to notebook --execute train_model_D_full.ipynb \
  --output train_model_D_full_done.ipynb --ExecutePreprocessor.timeout=-1 \
  --ExecutePreprocessor.kernel_name=tool26 > train_full_D_log.txt 2>&1 &
```

Progress: `tail -f train_full_D_log.txt` or `runs/ablation_split/training/ModelD_full_seed*/results.csv`.
Results: `full_D_validation.csv` and `sop_results_full_D/SOP_RESULTS.md`.
